### Introduction to Vehicle Routing Optimization (VRP) with NVIDIA cuOpt

NVIDIA cuOpt is widely recognized as the world's fastest logistics and decision optimization engine. While traditional CPU-based solvers can take minutes or even hours to calculate complex routing matrices as the number of stops increases, cuOpt leverages the massive parallel computing power of GPUs to discover globally optimized routes in a fraction of a second.
In this section of the notebook, we transition from conventional linear resource allocation models (MIP) to the Capacitated Vehicle Routing Problem (CVRP), applied directly to agricultural supply chain and crop logistics.

#### The Operational Scenario: Cooperative Crop Hauling

Imagine you are managing the harvest logistics for a centralized agricultural cooperative. The network consists of the following components:
1. The Central Silo (Depot): Located at Index 0, this is the mandatory starting point for all vehicles in the fleet and where they must return at the end of their collection runs.
2. Partner Farms (Nodes/Stops): A network of farms spread across the region, each with a specific quantity of grain bags ready for extraction (demanda_sacos).
3. The Homogeneous Fleet: A fleet of 3 identical trucks, where each truck has a strict structural payload constraint (capacidade_maxima = 150 bags).

 #### Internal Algorithm Architecture

The code executed below uses the high-performance native cuopt.routing API and operates across four fundamental pillars:

• Routing(num_locais, num_veiculos): Initializes the routing graph topology in memory, allocating data nodes for the farms and independent vehicle execution lines.

• set_matrix(matrix_custo.values): Automatically maps and transfers your Cost/Distance Matrix (calculated on the GPU using geoprocessing coordinates) directly into the CUDA Cores. The heuristic engine uses this matrix as its baseline to minimize physical distance.

• set_depot(0): Locks the geographic zero index as the fleet's terminal base, generating closed-loop constraint circuits.

• add_capacity_dimension: Enforces the physical boundaries of the CVRP. cuOpt dynamically monitors the cumulative load weight as a vehicle progresses along a path. If hitting the next farm would breach the truck's remaining payload capacity, the algorithm forces a return loop back to the silo and dispatches the next available truck.
With a computing time budget set to 5.0 seconds, the hardware-accelerated engine applies hybrid evolutionary algorithms to establish the absolute shortest total distance traveled (km), minimizing fuel consumption and machine operating hours.

##### Libraries

In [23]:
from cuopt import routing
import numpy as np 
import cudf
import cupy as cp 

##### Creating Synthetic Data and GPU Spatial Matrix Calculation

Before launching the routing solver, we need to establish our geographic space. In real-world agricultural logistics, spatial data is typically ingested from GPS coordinates (Latitude and Longitude) stored in databases or shapefiles.
This section generates a synthetic dataset simulating a rural region and computes the Distance Matrix entirely on the GPU, avoiding costly CPU-GPU memory bottlenecks.


 The Spatial Setup

1. The Central Depot (Silo): Fixed at Index 0 with coordinates (-22.90, -47.06), representing the physical hub of the cooperative.
2. The Farms: 14 randomized locations generated within a bounding box of roughly 45 km × 45 km around the hub, simulating a dense farming region.
3. The Yield Demand: Each farm is assigned a random supply requirement between 10 and 50 bags of grains. The depot is explicitly forced to 0 demand since it is the collection point.



Accelerated Matrix Math via CuPy and cuDF

Instead of calculating distances sequentially using slow Python loops, this script uses NVIDIA RAPIDS (cuDF & CuPy) to perform massive parallel vector math:

• Degree-to-Kilometer Transformation: Earth's curvature is approximated locally by multiplying coordinates by 111.0 (the average distance in kilometers per degree of latitude/longitude in this region of Brazil).

• Broadcasting Distance Matrix: The term coords_km[:, None, :] - coords_km[None, :, :] utilizes GPU memory broadcasting to compute the spatial delta between every single pair of locations simultaneously in a 3D matrix space, resolving the Euclidean distances (cp.sqrt) instantly.

In [24]:
# Initialize random seed for reproducibility
np.random.seed(42) 
n_fazendas = 15

In [32]:
#Generate random geographical coordinates representing the agricultural zone
lats = np.random.uniform(low=-23.10, high=-22.70, size=n_fazendas)
lons = np.random.uniform(low=-47.26, high=-46.86, size=n_fazendas)

# Override index 0 to act as our Central Silo (Depot)
lats[0], lons[0] = -22.90, -47.06 # Silo Central 

#Create the GPU DataFrame using RAPIDS cuDF
df_coords = cudf.DataFrame({'id_local': range(n_fazendas), 'latitude': lats, 'longitude': lons,
'bag_demand': np.random.randint(10, 50, size=n_fazendas) })

#The central depot requires zero pickup demand
df_coords.loc[0, 'demanda_sacos'] = 0 

In [33]:
#High-performance GPU function to calculate the cost matrix
def calcular_matriz_km(df):
    coords = cp.array(df[['latitude', 'longitude']].to_pandas().values)
    coords_km = coords * 111.0 
    dist_matrix = cp.sqrt(cp.sum((coords_km[:, None, :]- coords_km[None, :, :]) ** 2, axis=-1))
    return cudf.DataFrame(dist_matrix.get().astype(np.float32))

In [34]:
#Run the function to build our cost baseline
matrix_custo = calcular_matriz_km(df_coords)

In [35]:
print("--- GPU GEOPROCESSING SUCCESSFUL ---")
print(f"Distance Matrix Shape: {matrix_custo.shape} (15x15 nodes)")
print("Data structure is fully allocated on the GPU and optimized for cuOpt.")

--- GPU GEOPROCESSING SUCCESSFUL ---
Distance Matrix Shape: (15, 15) (15x15 nodes)
Data structure is fully allocated on the GPU and optimized for cuOpt.


In [36]:
matrix_custo

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,0.000000,27.166176,13.223196,8.746489,13.151246,14.623116,6.689750,17.040949,20.935562,20.084976,23.681578,25.770969,22.050459,22.292828,22.644464
1,27.166176,0.000000,31.447632,18.777958,28.369757,38.844437,21.225548,33.825947,22.249096,7.317816,33.636154,1.452299,44.036777,32.284389,37.076450
2,13.223196,31.447632,0.000000,18.145464,3.356734,25.182678,12.860011,3.828705,33.002407,24.344236,36.902252,30.356314,33.007435,35.504082,9.421707
3,8.746489,18.777958,18.145464,0.000000,16.424065,20.536919,5.333836,21.741274,15.252487,12.083222,22.098295,17.346779,26.715034,20.558084,26.846239
4,13.151246,28.369757,3.356734,16.424065,0.000000,26.420593,11.101645,5.648490,31.596670,21.375284,36.503098,27.325096,34.216778,35.050575,10.422387
5,14.623116,38.844437,25.182678,20.536919,26.420593,0.000000,20.989458,28.514904,23.656696,32.571121,18.659979,37.392139,7.826129,18.015661,33.906746
6,6.689750,21.225548,12.860011,5.333836,11.101645,20.989458,0.000000,16.415283,20.510561,13.953124,26.368628,19.897139,28.070099,24.855322,21.520924
7,17.040949,33.825947,3.828705,21.741274,5.648490,28.514904,16.415283,0.000000,36.729309,26.935524,40.722397,32.827663,36.315166,39.328075,5.624196
8,20.935562,22.249096,33.002407,15.252487,31.596670,23.656696,20.510561,36.729309,0.000000,19.818129,11.599016,21.049009,25.715803,10.409777,41.996059
9,20.084976,7.317816,24.344236,12.083222,21.375284,32.571121,13.953124,26.935524,19.818129,0.000000,30.253897,6.076839,38.376202,28.777868,30.594622


In [43]:
# 1. Setup problem sizing parameters from your synthetic generator
num_locations = len(df_coords)     # 15 locations total (1 Silo + 14 Farms)
num_vehicles = 3                   # Fleet size
num_orders = num_locations - 1     # 14 operational farm delivery orders
max_capacity = 150                 # Payload ceiling per truck


In [54]:
# 2. Extract demands and fleet constraints as native GPU structures (cudf)
farm_demands = cudf.Series(df_coords['bag_demand'].iloc[1:].values, dtype=np.int32)
fleet_capacities = cudf.Series([max_capacity] * num_vehicles, dtype=np.int32)


In [55]:
farm_demands

0     20
1     26
2     17
3     44
4     44
5     42
6     14
7     48
8     37
9     16
10    18
11    17
12    21
13    43
dtype: int32

In [56]:
fleet_capacities

0    150
1    150
2    150
dtype: int32

In [59]:
# 3. Initialize the official cuOpt DataModel
data_model = routing.DataModel(num_locations, num_vehicles, num_orders)

In [60]:
# 4. Map the matrix infrastructure on the GPU using the native method
data_model.add_cost_matrix(matrix_custo) 


In [61]:
# Define vehicle itineraries: departures and final destination returns (All tied to Silo 0)
silo_start = cudf.Series([0] * num_vehicles, dtype=np.int32)
silo_end = cudf.Series([0] * num_vehicles, dtype=np.int32)
data_model.set_vehicle_locations(silo_start, silo_end)

In [62]:
# Pass the operational customer destination coordinates (Indices 1 through 14)
farm_orders = cudf.Series(range(1, num_locations), dtype=np.int32)
data_model.set_order_locations(farm_orders)

In [63]:
# 5. Bind the physical payload capacity dimensions (CVRP)
data_model.add_capacity_dimension("bags", farm_demands, fleet_capacities)

In [64]:
# 6. Configure heuristic search limits directly from the routing client
settings = routing.SolverSettings()
settings.set_time_limit(5.0)  # Safe execution budget in seconds

In [65]:
# Launch the combinatorial parallel calculation on the GPU cores
harvest_solution = routing.Solve(data_model, settings)

In [70]:
# 7. Print the log parameters to the Jupyter console
print("--- HARVEST ROUTING RESULTS ---")
print(f"Solver Status: {harvest_solution.get_status()}")  # 0 indicates global success
total_distance = harvest_solution.get_total_objective()
print(f"Total Fleet Distance Traveled: {total_distance:.2f} km\n")
final_routes = harvest_solution.get_route()


--- HARVEST ROUTING RESULTS ---
Solver Status: 0
Total Fleet Distance Traveled: 174.34 km



In [71]:
for truck_id in range(num_vehicles):
    # Filter matrix paths mapped to this specific vehicle index
    # Note: cuOpt output dataframe maps vehicle references to the 'truck_id' column
    truck_filter = final_routes[final_routes['truck_id'] == truck_id]
    individual_route = truck_filter['location'].to_arrow().to_pylist()
    print(f"Truck {truck_id + 1} - Visitation Sequence: {individual_route}")

Truck 1 - Visitation Sequence: [0, 6, 9, 1, 11, 3, 0]
Truck 2 - Visitation Sequence: [0, 8, 13, 10, 12, 5, 0]
Truck 3 - Visitation Sequence: [0, 2, 7, 14, 4, 0]
